# Pakalorie YOLOv8 Food Classifier — Colab Training (CDX-006)

Trains `yolov8n-cls` on the merged 218-class Kaggle desi-food dataset (8,660 images) and produces every artifact the FYP report needs: `best.pt`, `best.onnx`, top-1/top-5 accuracy, a confusion matrix, an honest per-class error analysis, and a ready-to-paste model-card block.

**Before you start, you need:**
1. **GPU runtime** — menu: `Runtime > Change runtime type > T4 GPU`. The next cell refuses to continue without it.
2. **A Kaggle account + API token** — kaggle.com > your profile picture > *Settings* > *API* > *Create New Token*. This downloads `kaggle.json`. Keep it ready to upload.
3. **A GitHub token for the private repo** — github.com > *Settings* > *Developer settings* > *Personal access tokens* > *Fine-grained tokens* > generate one with **read-only Contents** access to `arhamhi/Pakalorie_FYP` (or ask Arham for one).
4. **~90 minutes** — setup ~10 min, training ~30–60 min on a free T4, eval ~5 min.

**If Colab disconnects mid-training:** reconnect, re-run every cell from the top EXCEPT the *Train (fresh start)* cell — use the *Resume after disconnect* cell instead. Checkpoints live on your Google Drive, so nothing is lost.

Run cells top to bottom with `Shift+Enter`.

In [ ]:
# 1. GPU check - stop early if the runtime has no GPU.
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
assert gpu.returncode == 0, 'No GPU. Fix: Runtime > Change runtime type > T4 GPU, then rerun.'
print('GPU OK:', gpu.stdout.strip())

In [ ]:
# 2. Install dependencies (~2 min). Safe to re-run.
%pip install -q ultralytics kaggle pillow pandas matplotlib seaborn scikit-learn onnx onnxslim onnxruntime
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# 3. Mount Google Drive - checkpoints and artifacts persist here across disconnects.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/pakalorie_ml')   # everything we keep
DRIVE_RUNS = DRIVE_BASE / 'runs'                            # training checkpoints
DRIVE_REPORTS = DRIVE_BASE / 'reports'                      # metrics + audit files
RUN_NAME = 'yolov8n_cls_218'                                # full 218-class baseline

ROOT = Path('/content/pakalorie-ml')                        # fast local scratch
RAW = ROOT / 'raw'
PREPARED = ROOT / 'food-cls-224'
for folder in (DRIVE_RUNS, DRIVE_REPORTS, RAW):
    folder.mkdir(parents=True, exist_ok=True)
print('Artifacts will persist in:', DRIVE_BASE)

## 4. Kaggle token

Run the next cell and upload the `kaggle.json` you downloaded from kaggle.com > Settings > API > *Create New Token*. It is placed where the Kaggle CLI expects it, with owner-only permissions.

In [ ]:
import os
kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(exist_ok=True)
if not (kaggle_dir / 'kaggle.json').exists():
    from google.colab import files
    print('Upload your kaggle.json now:')
    uploaded = files.upload()
    assert 'kaggle.json' in uploaded, 'You must upload the file named kaggle.json'
    (kaggle_dir / 'kaggle.json').write_bytes(uploaded['kaggle.json'])
    os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('Kaggle token in place.')

In [ ]:
# 5. Download both Kaggle datasets (~475 MB total, ~3 min). Safe to re-run (skips if present).
izbaiman = RAW / 'izbaiman-food-images'
useractivated = RAW / 'useractivated-dataset'
if not izbaiman.exists():
    !kaggle datasets download izbaiman/food-images -p {izbaiman} --unzip
if not useractivated.exists():
    !kaggle datasets download useractivated/dataset -p {useractivated} --unzip
!ls {RAW}

## 6. Get the repo scripts

The repo is private, so the next cell asks for a GitHub token (it is not stored in the notebook). If you have no token, ask Arham — or download the repo as a ZIP from GitHub in your browser, upload it with the Files sidebar, and unzip it to `/content/Pakalorie_FYP` instead.

In [ ]:
from getpass import getpass
REPO = Path('/content/Pakalorie_FYP')
if not (REPO / 'ml/scripts/prepare_dataset.py').exists():
    token = getpass('GitHub token (input hidden): ').strip()
    !git clone --depth 1 https://{token}@github.com/arhamhi/Pakalorie_FYP.git {REPO}
    del token
assert (REPO / 'ml/scripts/prepare_dataset.py').exists(), 'Repo scripts not found - clone failed?'
print('Repo scripts ready.')

In [ ]:
# 7. Prepare the dataset (~4 min): audit, SHA-1 exact dedupe, resize to 224px,
#    stratified 80/20 train/val split, Ultralytics folder layout.
#    Audit reports go to Drive so they survive the session.
!python {REPO / 'ml/scripts/prepare_dataset.py'} \
  --source izbaiman/food-images={izbaiman / 'dataset'} \
  --source useractivated/dataset={useractivated / 'dataset/dataset'} \
  --output {PREPARED} \
  --audit-json {DRIVE_REPORTS / 'dataset_audit.json'} \
  --audit-markdown {DRIVE_REPORTS / 'dataset_audit.md'} \
  --class-counts-csv {DRIVE_REPORTS / 'class_counts.csv'} \
  --imgsz 224 \
  --val-ratio 0.2 \
  --seed 42

n_train = sum(1 for _ in (PREPARED / 'train').rglob('*.jpg'))
n_val = sum(1 for _ in (PREPARED / 'val').rglob('*.jpg'))
n_classes = len([p for p in (PREPARED / 'train').iterdir() if p.is_dir()])
print(f'Prepared: {n_classes} classes | {n_train} train | {n_val} val images')

## 8. Train — fresh start

Full **218-class baseline** (decision logged 2026-06-10: honest baseline first, prune later only if needed). `yolov8n-cls`, 224px, 50 epochs, batch 64. Expect **~30–60 min on a free T4**; you will see one progress line per epoch.

Checkpoints are written to your Drive every epoch. This cell **refuses to run if a checkpoint already exists** so you don't overwrite a half-finished run — use the resume cell below instead.

In [ ]:
last_ckpt = DRIVE_RUNS / RUN_NAME / 'weights' / 'last.pt'
assert not last_ckpt.exists(), (
    f'A checkpoint already exists at {last_ckpt}.\n'
    'Run the "Resume after disconnect" cell instead, or change RUN_NAME in cell 3 for a fresh run.'
)
!python {REPO / 'ml/scripts/train.py'} --data {PREPARED} --model yolov8n-cls.pt \
  --epochs 50 --imgsz 224 --batch 64 --device 0 \
  --project {DRIVE_RUNS} --name {RUN_NAME}

In [ ]:
# 8b. RESUME after a disconnect - ONLY run this if training was interrupted.
#     (Re-run cells 1-7 first so the dataset exists again, then run this.)
from ultralytics import YOLO
last_ckpt = DRIVE_RUNS / RUN_NAME / 'weights' / 'last.pt'
assert last_ckpt.exists(), 'No checkpoint found - run the fresh-start training cell instead.'
YOLO(str(last_ckpt)).train(resume=True)

In [ ]:
# 9. Evaluate best.pt: top-1/top-5 + confusion matrix + eval_metrics.json (reproducible path).
best = DRIVE_RUNS / RUN_NAME / 'weights' / 'best.pt'
assert best.exists(), 'best.pt missing - did training finish?'
!python {REPO / 'ml/scripts/evaluate.py'} --weights {best} --data {PREPARED} --device 0 \
  --project {DRIVE_RUNS} --name {RUN_NAME}_eval --json-out {DRIVE_REPORTS / 'eval_metrics.json'}

from IPython.display import Image as IPyImage, display
import glob
cm = sorted(glob.glob(str(DRIVE_RUNS / f'{RUN_NAME}_eval*' / 'confusion_matrix_normalized.png')))
if cm:
    display(IPyImage(cm[-1], width=900))

In [ ]:
# 10. Honest error analysis: per-class recall from the confusion matrix,
#     worst 15 classes -> goes into MODELCARD.md as-is.
import numpy as np, pandas as pd
from ultralytics import YOLO

model = YOLO(str(best))
metrics = model.val(data=str(PREPARED), imgsz=224, batch=64, device=0, plots=False, verbose=False)
names = [metrics.names[i] for i in sorted(metrics.names)]
matrix = metrics.confusion_matrix.matrix  # rows = predicted, cols = true
support = matrix.sum(axis=0)
recall = np.divide(np.diag(matrix), support, out=np.zeros_like(support, dtype=float), where=support > 0)
per_class = pd.DataFrame({'class': names[:len(support)], 'val_images': support.astype(int), 'recall': recall.round(3)})
per_class.sort_values('recall').to_csv(DRIVE_REPORTS / 'per_class_recall.csv', index=False)
print(f'top-1 = {metrics.top1:.4f}   top-5 = {metrics.top5:.4f}')
print('\n15 weakest classes (likely under-sampled or noisy):')
print(per_class.sort_values('recall').head(15).to_string(index=False))

## 11. Qualitative test on our own photos

Upload a few of **Arham's held-out food photos** (real desi dishes, not from the training data). Top-5 predictions per photo go straight into the model card and the report.

In [ ]:
from google.colab import files
print('Upload your own food photos (jpg/png/webp):')
own = files.upload()
qual_lines = []
for fname in own:
    result = model.predict(fname, imgsz=224, verbose=False)[0]
    top5_idx = result.probs.top5
    top5 = [f'{result.names[i]} ({float(result.probs.data[i]):.2f})' for i in top5_idx]
    qual_lines.append('| ' + fname + ' | ' + top5[0] + ' | ' + ', '.join(top5[1:]) + ' |')
    print(fname + ': ' + ' | '.join(top5))
(DRIVE_REPORTS / 'qualitative_predictions.md').write_text(
    '| photo | top-1 (conf) | rest of top-5 |\n|---|---|---|\n' + '\n'.join(qual_lines) + '\n',
    encoding='utf-8')
print('\nSaved table to Drive: reports/qualitative_predictions.md')

In [ ]:
# 12. Generate the MODELCARD metrics block - paste this into ml/MODELCARD.md in the repo.
import json, datetime
eval_payload = json.loads((DRIVE_REPORTS / 'eval_metrics.json').read_text())
onnx_path = best.with_name('best.onnx')
onnx_note = f'present, {onnx_path.stat().st_size/1e6:.1f} MB' if onnx_path.exists() else 'MISSING - rerun train.py export'
block = (
    f'## Metrics (run {RUN_NAME}, {datetime.date.today().isoformat()})\n\n'
    '| Model | Classes | Top-1 | Top-5 | Epochs | imgsz |\n'
    '| --- | ---: | ---: | ---: | ---: | ---: |\n'
    f"| yolov8n-cls | {n_classes} | {eval_payload['top1']:.4f} | {eval_payload['top5']:.4f} | 50 | 224 |\n\n"
    f'- Confusion matrix: Drive `pakalorie_ml/runs/{RUN_NAME}_eval/confusion_matrix_normalized.png`\n'
    '- Per-class recall (worst-15 = honest error analysis): Drive `pakalorie_ml/reports/per_class_recall.csv`\n'
    '- Qualitative predictions on our own photos: Drive `pakalorie_ml/reports/qualitative_predictions.md`\n'
    f'- Weights: Drive `pakalorie_ml/runs/{RUN_NAME}/weights/best.pt` ({best.stat().st_size/1e6:.1f} MB)\n'
    f'- ONNX export: {onnx_note}\n'
)
(DRIVE_REPORTS / 'modelcard_block.md').write_text(block, encoding='utf-8')
print(block)

## 13. Done — what to send back

Everything below is already on your Google Drive under `pakalorie_ml/`. Share that folder (or these files) with Arham:

1. `runs/yolov8n_cls_218/weights/best.pt` and `best.onnx`
2. `reports/eval_metrics.json` (top-1/top-5)
3. `runs/yolov8n_cls_218_eval/confusion_matrix_normalized.png`
4. `reports/per_class_recall.csv` (worst-15 error analysis)
5. `reports/qualitative_predictions.md` (our own photos)
6. `reports/modelcard_block.md` (paste into `ml/MODELCARD.md`)

**If top-1 looks weak (< ~0.55):** that is still a reportable baseline — the dataset is noisy with 14.6x class imbalance and we say so honestly. A second run with `yolov8s-cls.pt` (change `--model` in cell 8 and `RUN_NAME` in cell 3 to `yolov8s_cls_218`) is the agreed next step, NOT class pruning.